# $\Lambda = 4, \lambda = 1.0$: EvolvedOperatorAnsatz

Notebook created by HLD for the work arXiv: 2503.13368 [quant-ph, hep-th]

In this notebook, we run the VQE experiments for bosonic SU(2) matrix model at $\Lambda = 4, \lambda = 1.0$ using different variants of EvolvedOperatorAnsatz with the fixed Estimator seed = 225.

In [1]:
import sys
sys.path.append('../../utility')
from vqe_run import *
from qc_ansatze import *
from L4_evop_func import *

In [2]:
l4_evop = L4_evop(1.0)

Min absolute value is 0.004487
Max absolute value is 1.75
Mean absolute value is 0.13128
None
E_exact = 3.52625


In [3]:
[print(f'{l4_evop.ansatz_names[i]}: {l4_evop.ansatz_list[i].num_parameters}') for i in range(len(l4_evop.ansatz_list))]

ev_op_Hp15: 15
ev_op_Hp20: 20
ev_op_Hp25: 25
ev_op_Hp30: 30
ev_op_H40: 40
ev_op_Hp15_2f: 30
ev_op_Hp20_2f: 40
ev_op_Hp25_2f: 50
ev_op_Hp30_2f: 60
ev_op_Hp40_2f: 80


[None, None, None, None, None, None, None, None, None, None]

In [5]:
H4q = l4_evop.H4q

# VQE individual runs

In [6]:
seed = 225
iterations = 650
algorithm_globals.random_seed = seed

#estimator
noiseless_estimator = AerEstimator(
    run_options={"seed": seed, "shots": 1024},
    transpile_options={"seed_transpiler": seed},
)
#storing values
counts = []
values = []
def store_intermediate_result(eval_count, parameters, mean, std):
    counts.append(eval_count)
    values.append(mean)
    
def run_qve_w_specified_optimizer(optimizer, ansatz):
    opt = optimizer(maxiter = iterations)
    vqe = VQE(noiseless_estimator, ansatz, optimizer=opt, callback=store_intermediate_result)
    result = vqe.compute_minimum_eigenvalue(operator=H4q).eigenvalue.real
    print(f"VQE result: {result:.5f}")
    return result

# COBYLA

In [7]:
r_cobyla=[]
for i in range(len(l4_evop.ansatz_list)):
    print(f'At step {i}, with {l4_evop.ansatz_names[i]}')
    counts = []
    values = []
    t0 = time.time()
    result = run_qve_w_specified_optimizer(COBYLA, l4_evop.ansatz_list[i])
    t1 = time.time()
    print(f'Length of this optimization {len(values)}, time taken = {np.round(t1-t0,3)} \n')
    counts_a = counts
    values_a = values 
    r_cobyla.append(pd.DataFrame({f'{l4_evop.ansatz_names[i]}': values_a}))
    

At step 0, with ev_op_Hp15
VQE result: 3.46287
Length of this optimization 178, time taken = 183.352 

At step 1, with ev_op_Hp20
VQE result: 3.42899
Length of this optimization 248, time taken = 266.831 

At step 2, with ev_op_Hp25
VQE result: 3.68217
Length of this optimization 304, time taken = 341.94 

At step 3, with ev_op_Hp30
VQE result: 3.77387
Length of this optimization 360, time taken = 430.5 

At step 4, with ev_op_H40
VQE result: 13.73346
Length of this optimization 453, time taken = 789.55 

At step 5, with ev_op_Hp15_2f
VQE result: 4.45527
Length of this optimization 332, time taken = 456.982 

At step 6, with ev_op_Hp20_2f
VQE result: 5.13299
Length of this optimization 423, time taken = 618.234 

At step 7, with ev_op_Hp25_2f
VQE result: 5.23314
Length of this optimization 538, time taken = 847.109 

At step 8, with ev_op_Hp30_2f
VQE result: 5.10160
Length of this optimization 650, time taken = 1113.674 

At step 9, with ev_op_Hp40_2f
VQE result: 15.28227
Length of thi

In [8]:
df1 = pd.concat([r_cobyla[i] for i in range(len(r_cobyla))], axis = 1)
df1.to_csv('results_seeds/l4_l10_op_ev_cobyla_seed225.csv')

# SPSA

In [9]:
r_spsa=[]
for i in range(len(l4_evop.ansatz_list)):
    print(f'At step {i}, with {l4_evop.ansatz_names[i]}')
    counts = []
    values = []
    t0 = time.time()
    result = run_qve_w_specified_optimizer(SPSA, l4_evop.ansatz_list[i])
    t1 = time.time()
    print(f'Length of this optimization {len(values)}, time taken = {np.round(t1-t0,3)} \n')
    counts_a = counts
    values_a = values 
    r_spsa.append(pd.DataFrame({f'{l4_evop.ansatz_names[i]}': values_a}))

At step 0, with ev_op_Hp15
VQE result: 3.67193
Length of this optimization 1351, time taken = 1576.071 

At step 1, with ev_op_Hp20
VQE result: 3.60327
Length of this optimization 1351, time taken = 1635.074 

At step 2, with ev_op_Hp25
VQE result: 3.49313
Length of this optimization 1351, time taken = 1707.627 

At step 3, with ev_op_Hp30
VQE result: 3.56695
Length of this optimization 1351, time taken = 1861.656 

At step 4, with ev_op_H40
VQE result: 3.57502
Length of this optimization 1351, time taken = 2357.805 

At step 5, with ev_op_Hp15_2f
VQE result: 3.51500
Length of this optimization 1351, time taken = 1951.998 

At step 6, with ev_op_Hp20_2f
VQE result: 3.59577
Length of this optimization 1351, time taken = 2112.427 

At step 7, with ev_op_Hp25_2f
VQE result: 3.68810
Length of this optimization 1351, time taken = 2154.187 

At step 8, with ev_op_Hp30_2f
VQE result: 3.63468
Length of this optimization 1351, time taken = 2413.193 

At step 9, with ev_op_Hp40_2f
VQE result: 5.

In [10]:
df2 = pd.concat([r_spsa[i] for i in range(len(r_spsa))], axis = 1)
df2.to_csv('results_seeds/l4_l10_op_ev_spsa_seed225.csv')